# Q-Learning Tabulaire : Frozen Lake et Taxi

Bienvenue lecteur dans le monde Fabuleux de l'Apprentissage par renforcement 😁​.  
Ce notebook explore le **Q-Learning**, un algorithme fondamental d'apprentissage par renforcement basé sur les valeurs. Nous l'appliquons à deux environnements discrets et de taille modérée : **Frozen Lake** (16 états) et **Taxi** (500 états).

L'objectif est triple :

1. comprendre le principe de base du Q-Learning : mémoriser la valeur de chaque paire (état, action) ;
2. implémenter soi-même l'algorithme avec une Q-table, sans framework IA ;
3. comparer avec une approche moderne utilisant Stable-Baselines3.

## Structure du notebook

- [**Partie 1 : Frozen Lake**](#2-frozen-lake--le-premier-environnement) — Introduction au Q-Learning sur un petit environnement.
- [**Partie 2 : Taxi**](#8-taxi--passage-à-un-espace-plus-grand) — Passage à 500 états et vérification que Q-Learning reste applicable.
- [**Partie 3 : Stable-Baselines**](#11-dqn--de-la-table-au-réseau) — Utilisation du DQN (extension du Q-Learning avec réseaux de neurones) pour montrer la progression depuis Q-Learning tabulaire.

Le Q-Learning fonctionne bien ici car les espaces d'états sont discrets et énumérables. Sur des espaces continus ou très grands, des approximateurs de fonction (réseaux de neurones) deviennent nécessaires.

Let's go 😝​

In [1]:
import random

import gymnasium as gym
import numpy as np

from stable_baselines3 import PPO, DQN, A2C, SAC 

import  pickle
from tqdm.notebook import tqdm

## 1. Dépendances et outils

Nous n'utilisons que **NumPy** pour les opérations matricielles et **Gymnasium** pour les environnements. Le Q-Learning tabulaire est suffisamment simple pour ne pas nécessiter PyTorch ou TensorFlow (Heureusement d'ailleurs 🫩​).

- **Gymnasium** : API standardisée pour les environnements.
- **NumPy** : manipulation de tableaux (Q-table).
- **tqdm** : barres de progression pour les boucles d'entraînement.
- **Pickle** : sérialisation de la Q-table.
- **Stable-Baselines3** : implémentations haute performance pour comparaison.

In [5]:
# Environnement

env = gym.make("FrozenLake-v1")

In [6]:
print(f"Espace d'observation: {env.observation_space}")

print(f"Espace d'action: {env.action_space}")

Espace d'observation: Discrete(16)
Espace d'action: Discrete(4)


## 2. Frozen Lake : Le premier environnement

### Espace d'observation et d'action

**Espace d'observation :** 16 états discrets correspondant aux 16 cases d'une grille 4×4. Chaque case est identifiée par un indice unique de 0 à 15.

**Espace d'action :** 4 actions discrètes :
- `0` : descendre (south)
- `1` : monter (north)
- `2` : aller à droite (east)
- `3` : aller à gauche (west)

### Objectif et récompenses

L'agent doit atteindre la case cible (position 15) sans tomber dans un trou (positions 5, 7, 11, 12). 

- Récompense `+1` en atteignant la cible.
- Récompense `0` en tombant dans un trou ou en quittant le plateau.
Notez qu'on peut personnaliser la récompense. Ici on peut configurer notre environnement  pour qu'il pénalise `-1` à chaque pas, encourageant une résolution rapide.

### Paramètre `is_slippery`

Par défaut, Frozen Lake est glissant : l'agent a 33 % de chance de glisser perpendiculairement à son action choisie. Ici, nous utilisons `is_slippery=False` pour un environnement déterministe, idéal pour débuter avec Q-Learning.

## 3. Q-Learning : Algorithme basé sur les valeurs

Le Q-Learning apprend une table `Q(s,a)` qui stocke la valeur attendue d'exécuter l'action `a` dans l'état `s`.

### Équation de mise à jour

À chaque pas, nous observons une transition $(s, a, r, s')$. La Q-table est mise à jour selon :

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \right]$$

où :

- $\alpha$ (learning_rate) : taux d'apprentissage (0,7 ici), contrôle la vitesse d'adaptation.
- $r$ : récompense immédiate obtenue.
- $\gamma$ (gamma) : facteur d'actualisation (0,95), importance des récompenses futures.
- $\max_{a'} Q(s',a')$ : meilleure valeur estimée pour l'état suivant.

Le terme entre crochets s'appelle l'**erreur de Bellman** : il mesure l'écart entre la valeur actuelle et l'estimation mise à jour.


PS: J'ai trouvé dans mes recherches une version bien plus intuitive de la formule

Q_new = Q_old + α[Target − Q_old]

Q_new = Q_old + αTarget −αQ_old

Donc :
Q_new=(1−α)Q_old + αTarget  (Magique pas vrai ?🧙​)


### Caractéristiques clés

- **Off-policy** : utilise la meilleure action observée, même si elle n'a pas été exécutée.
- **Convergence garantie** : avec les bonnes conditions, Q converge vers les vraies valeurs.
- **Tabulaire** : efficace pour espaces d'états petits et discrets.

## 4. Initialisation de la Q-table

La Q-table est une matrice de shape `(nombre_états, nombre_actions)`. Pour Frozen Lake, c'est une matrice 16×4. 

Nous l'initialisons avec des zéros : l'agent n'a aucune connaissance au départ et apprendra progressivement en explorant l'environnement.

```python
Q(state, action) = 0 pour tous (state, action)
```

Au fil de l'entraînement, les valeurs convergent vers les vraies valeurs espérées.

In [7]:
def initialize_q_table(state_space, action_space):
  Qtable = np.zeros((state_space , action_space ))
  return Qtable

## 5. Exploration vs Exploitation : Stratégie Epsilon-Greedy

Le Q-Learning apprend uniquement des états qu'il explore. Sans exploration, l'agent reste bloqué dans une trajectoire limitée et ne découvre jamais les bonnes actions.

### Stratégie Epsilon-Greedy

Avec une probabilité $\varepsilon$, l'agent **explore** en choisissant une action aléatoire. Sinon, il **exploite** en prenant l'action ayant la plus grande valeur Q :

$$a = \begin{cases}
\arg\max_{a'} Q(s, a') & \text{avec probabilité } 1 - \varepsilon \\
\text{action aléatoire} & \text{avec probabilité } \varepsilon
\end{cases}$$

Notez que $\arg\max$ est la caractéristique principale des algorithmes basés sur la valeur

### Décroissance d'Epsilon

Au début, $\varepsilon = 1$ (100 % d'exploration). Progressivement, $\varepsilon$ décroît vers une valeur minimale (0,05). Cela reflète l'idée que l'agent apprend et peut progressivement faire confiance à sa Q-table.

La décroissance exponentielle utilisée ici :

$$\varepsilon_e = \varepsilon_{\min} + (\varepsilon_{\max} - \varepsilon_{\min}) \cdot \exp(-\text{decay\_rate} \cdot e)$$

assure une transition lisse sans variation abrupte.

Mais bon, c'est une affaire de gout ou de couleur. Vous pouvez personnaliser la perte de $ \varepsilon$ comme vous voulez. Une petite boucle for et le tour est joué ​😼​

### Alternative : Softmax

Une autre stratégie d'exploration attribue une probabilité proportionnelle à la valeur Q. Les actions bonnes sont favorisées, mais pas exclusives. Nous utilisons ici epsilon-greedy pour sa simplicité.

In [8]:
# greedy
def greedy_policy(Qtable, state):
  action = np.argmax(Qtable[state][:])

  return action


In [9]:
def epsilon_greedy_policy(Qtable, state, epsilon):
  # Randomly generate a number between 0 and 1
  random_num = random.random()
  
  if random_num > epsilon:
    action = greedy_policy(Qtable, state)
  else:
    action = env.action_space.sample()
  return action

## 6. Boucle d'entraînement

L'entraînement suit le cycle standard :

1. **Réinitialiser** l'état initial d'un épisode.
2. **Choisir** une action selon epsilon-greedy.
3. **Exécuter** l'action dans l'environnement et observer $(r, s')$.
4. **Mettre à jour** la Q-table avec l'équation de Bellman.
5. **Passage** à l'état suivant et répétition.

La boucle extérieure répète ce processus sur plusieurs épisodes (100 000 ici pour Frozen Lake), tandis que la boucle interne exécute jusqu'à 99 pas par épisode avant arrêt (termination naturelle ou limite atteinte).

### Hyperparamètres clés

- `learning_rate = 0.7` : haut pour une convergence rapide sur petits espaces.
- `gamma = 0.95` : l'agent préfère les récompenses futures mais pas indéfiniment.
- `max_epsilon = 1.0, min_epsilon = 0.05` : plage de décroissance d'exploration.
- `decay_rate = 0.0005` : vitesse de décroissance (plus bas = décroissance plus lente).

Après l'entraînement, la Q-table contient les valeurs apprises et permet de générer une politique gloutonne : `a = argmax Q(s, :)`.

In [55]:
env = gym.make("FrozenLake-v1", is_slippery=False)

gamma = 0.95  
learning_rate = 0.7 
max_epsilon = 1.0       
min_epsilon = 0.05    
decay_rate = 0.0005  
max_steps = 99     

def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
  for episode in tqdm(range(n_training_episodes)):
    # Reduction de epsilon au fil de l'entrainement
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    
    
    state, info = env.reset()
    step = 0
    terminated = False
    truncated = False

    
    for step in range(max_steps):

      action = epsilon_greedy_policy(Qtable, state, epsilon)

    
      new_state, reward, terminated, truncated, _ = env.step(action)
      
      

      # Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
      Qtable[state][action] = Qtable[state][action] + learning_rate * (reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action])

 
      if terminated or truncated:
        break

  
      state = new_state
  return Qtable

### Entrainement

In [56]:
n_training_episodes = 100000

Qtable_frozenlake = initialize_q_table(16,4)
Qtable_frozenlake = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_frozenlake)

  0%|          | 0/100000 [00:00<?, ?it/s]

## 7. Évaluation de la politique apprise

Une fois l'entraînement terminé, nous évaluons la politique en mode déterministe : chaque état choisit l'action ayant la valeur Q maximale, sans exploration.

L'évaluation porte sur 100 épisodes indépendants pour calculer :

- **récompense moyenne** : nombre d'épisodes réussis (atteindre la cible).
- **écart-type** : variabilité d'une partie à l'autre.
- **taux de réussite** : pourcentage d'épisodes où la cible est atteinte.

Une politique apprenant correctement devrait avoir un taux de réussite de 100 % sur Frozen Lake déterministe, et une récompense moyenne de 1 (puisqu'une réussite donne +1).

In [57]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q):

  episode_rewards = []
  for episode in tqdm(range(n_eval_episodes)):
   
    state, info = env.reset()
    step = 0
    truncated = False
    terminated = False
    total_rewards_ep = 0

    for step in range(max_steps):
      # Take the action (index) that have the maximum expected future reward given that state
      action = greedy_policy(Q, state)
      new_state, reward, terminated, truncated, info = env.step(action)
      total_rewards_ep += reward

      if terminated or truncated:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

In [58]:
n_eval_episodes = 100


mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_frozenlake)
print(f"Récompense Moyenne ={mean_reward:.2f} +/- {std_reward:.2f}")

  0%|          | 0/100 [00:00<?, ?it/s]

Récompense Moyenne =1.00 +/- 0.00


### Victoire 🕺​🕺​🎉​🎉​🥳​

Un algorithme parfait.

## Observation

In [65]:
env = gym.make("FrozenLake-v1",is_slippery=False, render_mode="human")

obs, info = env.reset()

done = False
try :
    while not done:
        # Choisir la meilleure action selon la Q-table
        action = np.argmax(Qtable_frozenlake[obs])

        # Faire un pas dans l'environnement
        obs, reward, terminated, truncated, info = env.step(action)

        done = terminated or truncated

finally:
    env.close()

In [67]:
# Enregistrement

# Q-table
with open("models/q-table-frozenlake.pkl", "wb") as f:
   pickle.dump(Qtable_frozenlake , f)
   
# Enregistrement vidéo

from gymnasium.wrappers import RecordVideo

env = gym.make("FrozenLake-v1",is_slippery=False, render_mode="rgb_array")

env = RecordVideo(
    env, 
    video_folder="./videos", 
    episode_trigger=lambda ep_id: True,
    name_prefix="frozen-lake-scratch"
)

state, _ = env.reset()
done = False


try :
    while not done:
        # Choisir la meilleure action selon la Q-table
        action = np.argmax(Qtable_frozenlake[obs])

        # Faire un pas dans l'environnement
        obs, reward, terminated, truncated, info = env.step(action)

        done = terminated or truncated



finally:

    env.close()

print("Vidéo enregistrée dans le dossier ./videos")

e:\cours ifri\Programmation et BD\Python\Reinforcement Learning\HF Course\.venv\Lib\site-packages\gymnasium\wrappers\rendering.py:292: UserWarning: WARN: Overwriting existing videos at e:\cours ifri\Programmation et BD\Python\Reinforcement Learning\HF Course\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Vidéo enregistrée dans le dossier ./videos


## 8. Taxi : Passage à un espace plus grand

Après Frozen Lake (16 états), nous appliquons le même Q-Learning à **Taxi**, un environnement avec 500 états discrets.

### Espace d'observation

L'état encapsule :
- Position du taxi (25 positions possibles dans une grille 5×5).
- Position du passager (5 lieux prédéfinis + dans le taxi).
- Destination souhaitée (4 lieux).

Total : $25 \times 5 \times 4 = 500$ états.

### Actions

L'agent dispose de 6 actions :
- `0` : descendre (south)
- `1` : monter (north)
- `2` : aller à droite (east)
- `3` : aller à gauche (west)
- `4` : prendre le passager (pickup)
- `5` : déposer le passager (drop-off)

### Récompenses

- `-1` par étape.
- `+20` pour déposer le passager au bon endroit.
- `-10` pour une action illégale (ex. prendre un passager alors qu'il n'y en a pas).



### Même Q-Learning, mais plus grand

L'algorithme reste identique ; seule la taille de la Q-table change (500×6 au lieu de 16×4). Q-Learning n'a pas de problème avec cette augmentation car c'est une approche tabulaire.

On réutilisera nos implémentations et on verra bien ce que ça donne. Croisons les doigts ​👀​.

In [73]:
# Environnement

env = gym.make("Taxi-v4" , is_rainy = False)

In [74]:
print(f"Espace d'observation: {env.observation_space}")

print(f"Espace d'action: {env.action_space}")

Espace d'observation: Discrete(500)
Espace d'action: Discrete(6)


In [115]:
# Notre Q_table
env = gym.make("Taxi-v4" , is_rainy = False)
Qtable_taxi = initialize_q_table(500 , 6)

# Entrainement
print("Training")
Qtable_taxi = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_taxi)

# Evaluation
print("Evaluation")
mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_taxi)
print(f"Récompense Moyenne ={mean_reward:.2f} +/- {std_reward:.2f}")

Training


  0%|          | 0/100000 [00:00<?, ?it/s]

Evaluation


  0%|          | 0/100 [00:00<?, ?it/s]

Récompense Moyenne =7.70 +/- 2.56


In [ ]:
# On creuse un  peu pour observer les stats

n_eval_episodes_detailed_taxi = 100
success = 0
episode_rewards_taxi = []
episode_lengths_taxi = []

env = gym.make("Taxi-v4" , is_rainy = False)
for episode in range(n_eval_episodes_detailed_taxi):
    state, info = env.reset()
    step = 0
    truncated = False
    terminated = False
    total_reward = 0

    for step in range(max_steps):
        action = greedy_policy(Qtable_taxi, state)
        new_state, reward, terminated, truncated, info = env.step(action)
        total_reward += reward

        if reward ==20:
            success += 1
        if terminated or truncated:
            break
        state = new_state

    episode_rewards_taxi.append(total_reward)
    episode_lengths_taxi.append(step + 1)

rewards_taxi = np.array(episode_rewards_taxi, dtype=np.float32)
lengths_taxi = np.array(episode_lengths_taxi, dtype=np.int32)


print(f"=== TAXI - Évaluation sur 100 épisodes ===")
print(f"Récompense moyenne     : {rewards_taxi.mean():.2f}")
print(f"Récompense médiane     : {np.median(rewards_taxi):.2f}")
print(f"Écart-type             : {rewards_taxi.std():.2f}")
print(f"Meilleure récompense   : {rewards_taxi.max():.2f}")
print(f"Pire récompense        : {rewards_taxi.min():.2f}")
print(f"Nombre moyen de pas    : {lengths_taxi.mean():.2f}")
print(f"Livraisons réussies    : {success}/100")


=== TAXI - Évaluation sur 100 épisodes ===
Récompense moyenne     : 8.13
Récompense médiane     : 8.00
Écart-type             : 2.47
Meilleure récompense   : 14.00
Pire récompense        : 3.00
Nombre moyen de pas    : 12.87
Livraisons réussies    : 100/100


In [135]:
env = gym.make("Taxi-v4" , is_rainy = False , render_mode="human")

obs, info = env.reset()


done = False
try :
    while not done:
        # Choisir la meilleure action selon la Q-table
        action = np.argmax(Qtable_taxi[obs])

        # Faire un pas dans l'environnement
        obs, reward, terminated, truncated, info = env.step(action)

        done = terminated or truncated

finally:
    env.close()

In [81]:
# Enregistrement

# Q-table
with open("models/q-table-taxi.pkl", "wb") as f:
   pickle.dump(Qtable_taxi , f)
   
# Enregistrement vidéo

from gymnasium.wrappers import RecordVideo

env = gym.make("Taxi-v4" , is_rainy = False, render_mode="rgb_array")

env = RecordVideo(
    env, 
    video_folder="./videos", 
    episode_trigger=lambda ep_id: True,
    name_prefix="taxi-scratch"
)

state, _ = env.reset()
done = False


try :
    while not done:
        # Choisir la meilleure action selon la Q-table
        action = np.argmax(Qtable_taxi[obs])

        # Faire un pas dans l'environnement
        obs, reward, terminated, truncated, info = env.step(action)

        done = terminated or truncated



finally:

    env.close()

print("Vidéo enregistrée dans le dossier ./videos")

e:\cours ifri\Programmation et BD\Python\Reinforcement Learning\HF Course\.venv\Lib\site-packages\gymnasium\wrappers\rendering.py:292: UserWarning: WARN: Overwriting existing videos at e:\cours ifri\Programmation et BD\Python\Reinforcement Learning\HF Course\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Vidéo enregistrée dans le dossier ./videos


## 9. Limites du Q-Learning tabulaire

Bien que Q-Learning soit hyper beau et interprétable, il faut reconnaître les limitations :

- **Explosion combinatoire** : pour des espaces d'états continus ou très grands (10^6 états), une table devient impraticable (Je vous assure que vous ne voulez pas une Q-Table de 1 Milliard de case et un entrainement qui dure 1 semaine).
- **Pas de généralisation** : la Q-table n'apprend rien sur la structure de l'espace. Deux états similaires sont traités indépendamment (Un état qui est à 1.2 et un autre à 1.3 sont différents. Mais je vous laisse deviner pour qui ça ne l'est pas forcément 🌚​).
- **Maldiction de la dimensionnalité** : ajouter une dimension à l'observation multiplie le nombre d'états exponentiellement (Les plus malins auront compris que la complexité spatiale est de : $\mathcal{O}(|S| \times |A|)$)

**Solution** : approximer la Q-fonction avec un **réseau de neurones** qui généralise sur des états proches. C'est le principe du **Deep Q-Learning (DQN)**, qui remplace la table par un réseau $Q_\theta(s, a)$.
On verra l'implémentation en détail dans le notebook qui implémente Lunar Lander. Ici nous ferons juste une initiation.



## 10. Transition vers Stable-Baselines3

Comprendre les principes mathématiques sous-jacents est essentiel. Cependant, en pratique, on utilise des frameworks :

- **stabilité de l'entraînement** : les implémentations modernes gèrent les problèmes numériques, les normalisations d'états, etc.
- **efficacité** : elles exploitent le GPU et les vectorisations modernes.
- **reproductibilité** : elles utilisent des graines aléatoires contrôlées et des checklists de bonnes pratiques.
- **algorithmes avancés** : PPO, A3C, SAC et bien d'autres au-delà de DQN.

**Stable-Baselines3** est une excellente ressource : open-source, bien documentée et largement utilisée en recherche et en industrie.

Nous demonstrons le DQN (extension du Q-Learning avec approximation par réseau) sur Frozen Lake, montrant la progression naturelle du Q-Learning tabulaire vers le deep reinforcement learning.

## 11. DQN : De la table au réseau

Le **Deep Q-Network** remplace la Q-table par un réseau de neurones :

$$Q_\theta(s) = [Q_\theta(s, a_0), Q_\theta(s, a_1), \ldots, Q_\theta(s, a_n)]$$

Le réseau prend en entrée l'état et prédit un score pour chaque action. L'équation de mise à jour devient :

$$\text{Loss} = (Q_\theta(s, a) - [r + \gamma \max_{a'} Q_{\theta^-}(s', a')])^2$$

où $Q_{\theta^-}$ est un réseau "cible" gelé temporairement pour stabiliser l'entraînement.

### Avantages du DQN

- **Généralisation** : deux états similaires produisent des prédictions similaires.
- **Scalabilité** : fonctionne avec des espaces d'états continus ou énormes.
- **Flexibilité** : peut traiter différentes entrées (images, vecteurs, etc.).

Pour Frozen Lake, c'est overkill (une table suffit), mais c'est une bonne démonstration de la transition entre approches tabulaires et neuronales.

In [2]:
# On prend notre environnement
env = gym.make(
    "FrozenLake-v1",
    is_slippery=False # Ici on veut un en deterministe donc pas de sol glissant
)

### Instanciation du modèle

Stable-Baselines3 utilise une API simple et cohérente :

1. créer l'environnement ;
2. instancier l'agent (ici `DQN`) avec une politique réseau et des hyperparamètres ;
3. appeler `model.learn(total_timesteps)` ;
4. évaluer sur des épisodes de test.

Les hyperparamètres `gamma` et `learning_rate` sont similaires à ceux du Q-Learning, mais le réseau ajoute des paramètres supplémentaires (taille du buffer, synchronisation des réseaux cible, etc.) gérés automatiquement.

La politique `"MlpPolicy"` (Multi-Layer Perceptron) signifie que le réseau a une architecture simple : couches entièrement connectées.

In [ ]:
# On instancie notre modéle évec quelques hyperparamètres
env = gym.make("FrozenLake-v1", is_slippery=False)
model = DQN(
    "MlpPolicy", # Multi Layer Perceptron ( CnnPolicy pour des images )
 env,
    gamma=0.99,
    learning_rate=0.001,
    learning_starts=1000,
    exploration_fraction=0.8,
    target_update_interval=250 # Intervalle à laquelle on copie les poids du réseau à la cible
    )

### Petite anecdote

Le paramètre `target_update_interval` m'a volé des nuits 😭​. En gros il nous donne à quel intervalle on copie les poids du réseau vers la cible (Vous verrez tout ceci en détail promis). Par défaut il était réglé sur 10_000 alors moi, mon entrainement ... c'était 10_000. Donc mon réseau ne faisait presque jamais d'update et visait un cible avec des paramètres initialisés aléatoirement.

Toujours rester concentré dans la vie 🙂‍↕️​

In [16]:
# Entrainement
model.learn(total_timesteps=50000 , progress_bar=True)

Output()

In [22]:
# Évaluation détaillée du DQN sur Frozen Lake

dqn_eval_env = gym.make("FrozenLake-v1", is_slippery=False)
max_steps = 99    
n_eval_dqn = 100
episode_rewards_dqn = []
episode_lengths_dqn = []

for episode in range(n_eval_dqn):
    obs, info = dqn_eval_env.reset()
    done = False
    episode_reward = 0
    steps = 0
    
    while not done and steps < max_steps:
        action, _ = model.predict(obs, deterministic=True)
        # Convertir l'action en entier (elle est retournée comme np.ndarray)
        action = int(action)
        obs, reward, terminated, truncated, info = dqn_eval_env.step(action)
        episode_reward += reward
        done = terminated or truncated
        steps += 1
    
    episode_rewards_dqn.append(episode_reward)
    episode_lengths_dqn.append(steps)

dqn_eval_env.close()

rewards_dqn = np.array(episode_rewards_dqn, dtype=np.float32)
lengths_dqn = np.array(episode_lengths_dqn, dtype=np.int32)
successes_dqn = int(np.sum(rewards_dqn > 0))

print(f" DQN  - Évaluation sur 100 épisodes ")
print(f"Récompense moyenne     : {rewards_dqn.mean():.2f}")
print(f"Récompense médiane     : {np.median(rewards_dqn):.2f}")
print(f"Écart-type             : {rewards_dqn.std():.2f}")
print(f"Meilleure récompense   : {rewards_dqn.max():.2f}")
print(f"Pire récompense        : {rewards_dqn.min():.2f}")
print(f"Nombre moyen de pas    : {lengths_dqn.mean():.2f}")
print(f"Atterrissages réussis  : {successes_dqn}/100")


 DQN  - Évaluation sur 100 épisodes 
Récompense moyenne     : 1.00
Récompense médiane     : 1.00
Écart-type             : 0.00
Meilleure récompense   : 1.00
Pire récompense        : 1.00
Nombre moyen de pas    : 6.00
Atterrissages réussis  : 100/100


### Résultats

Réseau tout aussi parfait ✅. Sans surprise d'ailleurs. 

In [21]:
env_print = gym.make("FrozenLake-v1" , is_slippery=False, render_mode="human")

obs, info = env_print.reset()


done = False
try :
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        action = int(action)
        
        # Faire un pas dans l'environnement
        obs, reward, terminated, truncated, info = env_print.step(action)

        done = terminated or truncated

finally:
    env_print.close()

## 12. Conclusion : Du Q-Learning au Deep RL

Ce notebook a montré trois approches progressives, du plus simple au plus complexe, avec un tableau comparatif des résultats empiriques.

### Résultats empiriques : Tableau comparatif

| Approche | Environnement | Taille espace | Récompense moy. | Taux réussite |
|----------|---------------|---|---|---|
| Q-Learning tabulaire | Frozen Lake | 16 états | 1.00 | 100% |
| Q-Learning tabulaire | Taxi | 500 états | 7.69 | 100% |
| DQN (Stable-Baselines3) | Frozen Lake | 16 états | 1 | 100% |

### Analyse

1. **Q-Learning tabulaire sur Frozen Lake** : succès total.
   - Petit espace (16 états) 
   - Apprentissage rapide et convergence.
   
2. **Q-Learning tabulaire sur Taxi** : succès total.
   - Espace plus grand (500 états)
   - Apprentissage toujours aussi facile et bonne convergence.
 
   
3. **DQN sur Frozen Lake** :succès total.
   - Mettre les bons hyperparamètres est vitale en RL sinon meme avec des dizaines d'itérations on peut ne pas atteindre la convergence.

### Leçons clés

- Le **Q-Learning tabulaire** est simple, exact, mais limité à des espaces petits et discrets.
- Les **réseaux de neurones** (DQN) offrent une voie vers la scalabilité, mais nécessitent un tuning empirique important.
- Les métriques d'évaluation (récompense moyenne, taux de réussite sur 100 épisodes) sont **essentielles** pour juger objectivement la qualité d'une politique.
- Aucun algorithme ne marche "out-of-the-box" : **l'expérimentation empirique est au cœur du RL**.


Thanks you ​🩵